# 04 — Modelo predictivo con machine learning

**Proyecto:** Vigía Cali — sistema auditable de vigilancia temporal de la criminalidad reportada para la planeación institucional.

**Autores:** completar manualmente antes de la entrega.

## Alcance

Este cuaderno entrena y evalúa modelos únicamente cuando los datos superan
las puertas de calidad. La pregunta predictiva es: **¿puede machine learning
mejorar el pronóstico de homicidios reportados del próximo mes frente a repetir
el valor del mismo mes del año anterior?**

**Objetivo candidato:** cantidad mensual reportada de homicidios en
Cali. Se elige por tener una fuente principal separada y evitar el
solapamiento de Hurto por Modalidades.

**Diferenciación del producto:** el objetivo es anticipar la carga
mensual reportada a escala de ciudad para seguimiento y planeación temporal.
No se estima riesgo por comuna, barrio o persona, y una predicción no se
convierte automáticamente en una decisión de despliegue o presupuesto.


## 1. Configuración y serie candidata


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json
import os
import re
import unicodedata

os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.base import clone
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import PoissonRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

DATAV3_ROOT = Path("/content/drive/MyDrive/datav3")
SIEDCO_INPUT = DATAV3_ROOT / "A1 - SIEDCO" / "datos_criminalidad_cali"
PROJECT_OUTPUT = DATAV3_ROOT / "project_diplodata_outputs" / "eda_01_05_v1"
LANDING = PROJECT_OUTPUT / "landing"
TRUSTED = PROJECT_OUTPUT / "trusted"
SURFACE = PROJECT_OUTPUT / "surface"
AUDIT = PROJECT_OUTPUT / "audit"
REPORTS = PROJECT_OUTPUT / "reportes"

for directory in (LANDING, TRUSTED, SURFACE, AUDIT, REPORTS):
    directory.mkdir(parents=True, exist_ok=True)

PERIODO_INICIO = 2018
PERIODO_FIN = 2025
MODEL_START = pd.Timestamp(f"{PERIODO_INICIO}-01-01")
MODEL_END = pd.Timestamp(f"{PERIODO_FIN}-12-01")
TARGET = "cantidad_mensual_homicidios"
RANDOM_STATE = 42

sns.set_theme(style="whitegrid", context="notebook")

data_path = SURFACE / "analitica_eda.csv"
if not data_path.is_file():
    raise FileNotFoundError("Ejecute los notebooks 00–03 antes de evaluar modelado.")

data = pd.read_csv(data_path, low_memory=False)
data["fecha"] = pd.to_datetime(data["fecha"], errors="coerce")
data["cantidad"] = pd.to_numeric(data["cantidad"], errors="coerce")

homicide_rows = data.loc[data["tipo_delito"].eq("HOMICIDIO")].copy()
invalid_dates = int(homicide_rows["fecha"].isna().sum())
invalid_quantities = int(
    (homicide_rows["cantidad"].isna() | homicide_rows["cantidad"].lt(0)).sum()
)
if "flag_duplicado_exacto" in homicide_rows.columns:
    unresolved_duplicates = int(
        homicide_rows["flag_duplicado_exacto"].fillna(False).astype(bool).sum()
    )
else:
    unresolved_duplicates = None

homicides = homicide_rows.loc[
    homicide_rows["fecha"].between(MODEL_START, MODEL_END + pd.offsets.MonthEnd(1))
    & homicide_rows["cantidad"].notna()
    & homicide_rows["cantidad"].ge(0)
].copy()
monthly_target = (
    homicides.assign(mes=lambda frame: frame["fecha"].dt.to_period("M").dt.to_timestamp())
    .groupby("mes", as_index=False)["cantidad"].sum(min_count=1)
    .rename(columns={"cantidad": TARGET})
    .sort_values("mes")
)
display(monthly_target.head())
display(monthly_target.tail())


## 2. Criterios de habilitación

**Comprobaciones:** cantidades y fechas válidas, 96 meses completos entre
enero de 2018 y diciembre de 2025, duplicados exactos resueltos y auditoría
disponible. Un mes ausente no se rellena automáticamente con cero porque
puede significar falta de cobertura. Si una puerta falla, el entrenamiento
se detiene con un mensaje explícito.


In [ ]:
if monthly_target.empty:
    raise RuntimeError("No hay serie mensual de homicidios.")

expected_months = pd.date_range(MODEL_START, MODEL_END, freq="MS")
missing_months = expected_months.difference(monthly_target["mes"])
extra_months = monthly_target.loc[
    ~monthly_target["mes"].isin(expected_months), "mes"
]
gates = pd.DataFrame(
    [
        {
            "criterio": "Fechas de homicidios válidas",
            "cumple": invalid_dates == 0,
            "evidencia": f"{invalid_dates} fechas inválidas",
        },
        {
            "criterio": "Cantidades de homicidios válidas",
            "cumple": invalid_quantities == 0,
            "evidencia": f"{invalid_quantities} cantidades inválidas",
        },
        {
            "criterio": "Cobertura mensual completa 2018–2025",
            "cumple": len(missing_months) == 0 and extra_months.empty,
            "evidencia": (
                f"{len(monthly_target)}/96 meses; faltantes: "
                + (", ".join(month.strftime("%Y-%m") for month in missing_months) or "ninguno")
            ),
        },
        {
            "criterio": "Duplicados exactos revisados",
            "cumple": unresolved_duplicates == 0,
            "evidencia": (
                "bandera ausente" if unresolved_duplicates is None
                else f"{unresolved_duplicates} filas marcadas"
            ),
        },
        {
            "criterio": "Auditoría de calidad disponible",
            "cumple": (AUDIT / "calidad_por_fuente.csv").is_file(),
            "evidencia": str(AUDIT / "calidad_por_fuente.csv"),
        },
    ]
)
display(gates)
gates.to_csv(
    AUDIT / "puerta_modelado_homicidios.csv", index=False, encoding="utf-8-sig"
)
if not gates["cumple"].all():
    failed = gates.loc[~gates["cumple"], "criterio"].tolist()
    raise RuntimeError("MODELADO BLOQUEADO: " + "; ".join(failed))


## 3. Variables predictoras y modelos

Se formula un pronóstico **un mes adelante**. Las variables predictoras usan
solo información conocida antes del mes objetivo. La evaluación simula una
actualización mensual: para pronosticar cada mes ya se conoce el valor real del
mes anterior.

- rezagos de 1, 2, 3, 6 y 12 meses;
- medias móviles de 3, 6 y 12 meses, desplazadas un período;
- mes del año mediante seno y coseno;
- tendencia temporal.

Se comparan dos modelos de machine learning —regresión de Poisson regularizada
y bosque aleatorio— contra el baseline estacional `t-12`. La validación usa
ventanas expansivas, sin mezcla aleatoria. Los primeros bloques seleccionan
el mejor modelo ML y el último bloque de 12 meses funciona como prueba final.
Las métricas son MAE, RMSE y MASE; valores menores son mejores.


In [ ]:
def build_features(monthly_frame):
    featured = monthly_frame.sort_values("mes").copy()
    for lag in (1, 2, 3, 6, 12):
        featured[f"rezago_{lag}"] = featured[TARGET].shift(lag)
    for window in (3, 6, 12):
        featured[f"media_movil_{window}"] = (
            featured[TARGET].shift(1).rolling(window=window).mean()
        )
    featured["mes_seno"] = np.sin(2 * np.pi * featured["mes"].dt.month / 12)
    featured["mes_coseno"] = np.cos(2 * np.pi * featured["mes"].dt.month / 12)
    featured["tendencia"] = np.arange(len(featured), dtype=float)
    return featured

feature_columns = [
    "rezago_1", "rezago_2", "rezago_3", "rezago_6", "rezago_12",
    "media_movil_3", "media_movil_6", "media_movil_12",
    "mes_seno", "mes_coseno", "tendencia",
]
model_data = build_features(monthly_target).dropna(
    subset=feature_columns + [TARGET]
).reset_index(drop=True)
if len(model_data) < 72:
    raise RuntimeError("Se requieren al menos 72 filas supervisadas para dos validaciones y una prueba.")

candidate_models = {
    "Poisson regularizada": Pipeline(
        [
            ("escalado", StandardScaler()),
            ("modelo", PoissonRegressor(alpha=1.0, max_iter=2000)),
        ]
    ),
    "Bosque aleatorio": RandomForestRegressor(
        n_estimators=400,
        max_depth=5,
        min_samples_leaf=3,
        max_features=0.8,
        random_state=RANDOM_STATE,
        n_jobs=1,
    ),
}
baseline_name = "Baseline estacional (t-12)"
initial_train = 48
horizon = 12
metric_rows = []
prediction_rows = []
split_rows = []
train_end = initial_train
fold = 1

while train_end + horizon <= len(model_data):
    train = model_data.iloc[:train_end]
    validation = model_data.iloc[train_end:train_end + horizon]
    validation_start = validation.iloc[0]["mes"]
    history = monthly_target.loc[
        monthly_target["mes"].lt(validation_start), TARGET
    ].to_numpy(dtype=float)
    seasonal_scale = np.mean(np.abs(history[12:] - history[:-12]))
    if not np.isfinite(seasonal_scale) or seasonal_scale <= 0:
        raise RuntimeError("No se puede calcular MASE: escala estacional nula o inválida.")

    fold_predictions = {
        baseline_name: validation["rezago_12"].to_numpy(dtype=float)
    }
    for model_name, estimator in candidate_models.items():
        fitted = clone(estimator).fit(train[feature_columns], train[TARGET])
        fold_predictions[model_name] = np.clip(
            fitted.predict(validation[feature_columns]), 0, None
        )

    split_rows.append(
        {
            "fold": fold,
            "entrenamiento_inicio": train.iloc[0]["mes"],
            "entrenamiento_fin": train.iloc[-1]["mes"],
            "validacion_inicio": validation.iloc[0]["mes"],
            "validacion_fin": validation.iloc[-1]["mes"],
        }
    )
    for model_name, predictions in fold_predictions.items():
        actual = validation[TARGET].to_numpy(dtype=float)
        mae = mean_absolute_error(actual, predictions)
        rmse = np.sqrt(mean_squared_error(actual, predictions))
        metric_rows.append(
            {
                "fold": fold,
                "modelo": model_name,
                "mae": mae,
                "rmse": rmse,
                "mase": mae / seasonal_scale,
            }
        )
        for month, observed, predicted in zip(validation["mes"], actual, predictions):
            prediction_rows.append(
                {
                    "fold": fold,
                    "mes": month,
                    "modelo": model_name,
                    "real": observed,
                    "prediccion": predicted,
                }
            )
    fold += 1
    train_end += horizon

metrics = pd.DataFrame(metric_rows)
predictions_cv = pd.DataFrame(prediction_rows)
split_plan = pd.DataFrame(split_rows)
if split_plan["fold"].nunique() < 3:
    raise RuntimeError("Se requieren dos ventanas de selección y una prueba final.")

test_fold = int(split_plan["fold"].max())
metrics["uso"] = np.where(metrics["fold"].eq(test_fold), "PRUEBA_FINAL", "SELECCION")
predictions_cv["uso"] = np.where(
    predictions_cv["fold"].eq(test_fold), "PRUEBA_FINAL", "SELECCION"
)
selection_summary = (
    metrics.loc[metrics["uso"].eq("SELECCION")]
    .groupby("modelo", as_index=False)[["mae", "rmse", "mase"]]
    .mean()
    .sort_values("mase")
)
best_ml_name = selection_summary.loc[
    selection_summary["modelo"].isin(candidate_models), "modelo"
].iloc[0]
test_summary = metrics.loc[metrics["uso"].eq("PRUEBA_FINAL")].sort_values("mase")
best_ml_test_mase = float(
    test_summary.loc[test_summary["modelo"].eq(best_ml_name), "mase"].iloc[0]
)
baseline_test_mase = float(
    test_summary.loc[test_summary["modelo"].eq(baseline_name), "mase"].iloc[0]
)
ml_adds_value = best_ml_test_mase < baseline_test_mase
operational_model = best_ml_name if ml_adds_value else baseline_name

display(split_plan)
display(selection_summary)
display(test_summary)
print(f"Mejor modelo ML bloqueado antes de prueba: {best_ml_name}")
print(f"¿Supera al baseline en prueba final?: {'Sí' if ml_adds_value else 'No'}")

plt.figure(figsize=(9, 5))
sns.barplot(data=test_summary, x="mae", y="modelo", color="#4472C4")
plt.title("Error absoluto medio en la prueba final")
plt.xlabel("MAE (homicidios reportados por mes)")
plt.ylabel("Modelo")
plt.tight_layout()
plt.savefig(REPORTS / "modelo_mae_prueba_final.png", dpi=150, bbox_inches="tight")
plt.show()

test_plot = predictions_cv.loc[
    predictions_cv["uso"].eq("PRUEBA_FINAL")
    & predictions_cv["modelo"].isin([baseline_name, best_ml_name])
]
plt.figure(figsize=(11, 5))
sns.lineplot(data=test_plot, x="mes", y="prediccion", hue="modelo", marker="o")
actual_test = test_plot[["mes", "real"]].drop_duplicates()
plt.plot(actual_test["mes"], actual_test["real"], color="black", marker="o", label="Real")
plt.title("Homicidios reportados: predicción un mes adelante")
plt.xlabel("Mes")
plt.ylabel("Homicidios reportados por mes")
plt.legend(title="Serie")
plt.tight_layout()
plt.savefig(REPORTS / "modelo_predicciones_prueba_final.png", dpi=150, bbox_inches="tight")
plt.show()

future_month = monthly_target["mes"].max() + pd.offsets.MonthBegin(1)
future_base = pd.concat(
    [
        monthly_target,
        pd.DataFrame({"mes": [future_month], TARGET: [np.nan]}),
    ],
    ignore_index=True,
)
future_features = build_features(future_base).iloc[[-1]]
if future_features[feature_columns].isna().any(axis=None):
    raise RuntimeError("No fue posible construir las variables del próximo mes.")
if operational_model == baseline_name:
    next_prediction = float(future_features.iloc[0]["rezago_12"])
else:
    final_estimator = clone(candidate_models[operational_model]).fit(
        model_data[feature_columns], model_data[TARGET]
    )
    next_prediction = float(
        np.clip(final_estimator.predict(future_features[feature_columns])[0], 0, None)
    )

operational_residuals = predictions_cv.loc[
    predictions_cv["modelo"].eq(operational_model), "real"
].to_numpy() - predictions_cv.loc[
    predictions_cv["modelo"].eq(operational_model), "prediccion"
].to_numpy()
lower_error, upper_error = np.quantile(operational_residuals, [0.10, 0.90])
next_forecast = pd.DataFrame(
    {
        "mes_pronosticado": [future_month],
        "modelo_operativo": [operational_model],
        "prediccion_homicidios_reportados": [next_prediction],
        "limite_inferior_80": [max(0.0, next_prediction + lower_error)],
        "limite_superior_80": [max(0.0, next_prediction + upper_error)],
        "ml_supera_baseline_prueba": [ml_adds_value],
    }
)
display(next_forecast)

split_plan.to_csv(AUDIT / "plan_validacion_temporal_homicidios.csv", index=False, encoding="utf-8-sig")
metrics.to_csv(REPORTS / "metricas_modelos_homicidios.csv", index=False, encoding="utf-8-sig")
predictions_cv.to_csv(REPORTS / "predicciones_validacion_homicidios.csv", index=False, encoding="utf-8-sig")
next_forecast.to_csv(REPORTS / "pronostico_siguiente_mes_homicidios.csv", index=False, encoding="utf-8-sig")
model_card = {
    "objetivo": "homicidios reportados del próximo mes en Cali",
    "unidad": "homicidios reportados por mes",
    "periodo_datos": f"{MODEL_START:%Y-%m} a {MODEL_END:%Y-%m}",
    "mejor_modelo_ml_seleccion": best_ml_name,
    "modelo_operativo": operational_model,
    "ml_supera_baseline_prueba_final": bool(ml_adds_value),
    "intervalo": "empírico 80% con residuos fuera de muestra; no es garantía",
    "uso": "apoyar revisión y planeación mensual",
    "no_uso": "no asignar recursos automáticamente ni inferir riesgo territorial o individual",
}
with (REPORTS / "ficha_modelo_homicidios.json").open("w", encoding="utf-8") as handle:
    json.dump(model_card, handle, ensure_ascii=False, indent=2)


## 4. Interpretación y decisión

El notebook **sí entrena machine learning** cuando todas las puertas se
cumplen. La regresión de Poisson es apropiada para una variable de conteo y
el bosque aleatorio permite representar relaciones no lineales. Ninguno se
acepta por su nombre: el mejor modelo ML se elige sin mirar la prueba final y
solo se usa si supera al baseline estacional en esa prueba.

La salida incluye la unidad **homicidios reportados por mes**, un intervalo
empírico del 80%, métricas fuera de muestra y una ficha de limitaciones. Si
machine learning no mejora el baseline, el resultado correcto es conservar el
baseline y declarar que no se demostró valor predictivo adicional. La decisión
institucional sigue siendo humana y no se infiere riesgo por comuna, barrio o
persona.
